In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()


In [ ]:
def mean_pooling(model_output, attention_mask):
    """
    Функция, которая из выхода трансформера (по токенам)
    сделает один вектор на текст - усреднением по токенам.
    """
    # извлекаем эмбеддинги каждого токена на последнем слое модели
    token_embeddings = model_output.last_hidden_state  # (B, T, H): B=batch, T=tokens, H=hidden
    # растягиваем маску по скрытому размеру, чтобы совпала с (B, T, H):
    attention_mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    # сумма эмбеддингов реальных токенов каждого текста
    summed = (token_embeddings * attention_mask).sum(dim=1) # (B, H)
    # Сколько реальных токенов у каждого текста
    counts = attention_mask.sum(dim=1).clamp(min=1e-9) # (B, 1)
    return summed / counts # (B, H)


In [ ]:
def build_rubert_embeddings(
    texts,
    batch_size: int = 32,
    max_length: int = 256,
):
  """
  Функция принимает список текстов и возвращает матрицу эмбеддингов
  """  
    all_embeddings = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]

            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True, # обрезает длинные тексты
                max_length=max_length, # макс длина в токенах
                return_tensors="pt" # вернуть PyTorch tensors
            )

            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            outputs = model(**enc) # (B, T, H)
            emb = mean_pooling(outputs, enc["attention_mask"])  # (B, H)

            all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings) # (N, hidden)


In [ ]:
train = pd.read_parquet("/kaggle/input/table-dataset-exam-avito/train.parquet")
test = pd.read_parquet("/kaggle/input/table-dataset-exam-avito/test.parquet")

train["text_join"] = (train["title"] + " " + train["description"])
test["text_join"] = (test["title"] + " " + test["description"])

X_train_text = train["text_join"].fillna("").astype(str).tolist()
X_test_text = test["text_join"].fillna("").astype(str).tolist()

train_rubert_emb = build_rubert_embeddings(
    X_train_text,
    batch_size=32,
    max_length=256
)

test_rubert_emb = build_rubert_embeddings(
    X_test_text,
    batch_size=32,   
    max_length=256
)

train_rubert_emb.shape

In [ ]:
# save parquet with item_id
emb_cols = [f"text_emb_{j}" for j in range(train_rubert_emb.shape[1])]

train_text_emb_df = pd.DataFrame(train_rubert_emb, columns=emb_cols)
train_text_emb_df.insert(0, "item_id", train["item_id"])

test_text_emb_df = pd.DataFrame(test_rubert_emb, columns=emb_cols)
test_text_emb_df.insert(0, "item_id", test["item_id"])

out_path = "/kaggle/working/"
train_path = os.path.join(out_path, "RuBert_train.parquet")
test_path  = os.path.join(out_path, "RuBert_test.parquet")

train_text_emb_df[emb_cols] = train_text_emb_df[emb_cols].astype("float32")
test_text_emb_df[emb_cols]  = test_text_emb_df[emb_cols].astype("float32")

train_text_emb_df.to_parquet(train_path, index=False)
test_text_emb_df.to_parquet(test_path, index=False)

print("Saved:", train_path, test_path)